In [ ]:
library(Seurat)
# library(SeuratDisk)

library(reticulate)
library(anndata)

library(PRROC)
library(Matrix)
library(MASS)
library(dplyr)
library(tidyr)
library(repr)
library(reshape2)

library(UpSetR)
library(grid)
library(clustree)
library(pheatmap)

library(ggplot2)
library(ggpubr)
library(RColorBrewer)
library(cowplot)
library(hexbin)
library(scico)

library(lme4)
library(broom)
library(broom.mixed)
library(ggeffects)
library(geepack)

options(repr.plot.width=10, repr.plot.height=8)

getwd()

dataset_id <- "simulated_mm_RA"
genome_id <- "mm10"
samples <- c("all", "old", "young")
TEsubset <- "TP"

for (sample in samples) {
    dir.create(paste0("figures_", dataset_id, "_", sample))
}

colorTools <- c(
    "STARsolo_TE" = "#FFBC81",
    "STARsolo_TE_EM" = "#F3E088",
    "SoloTE_unique" = "#A4DD9B",
    "SoloTE_thr2" = "#6FC69D",
    "SoloTE_thr1" = "#4BB2BB",
    "SoloTE_thr0" = "#3989BF",
    "Stellarscope" = "#4F5D93",
    "simulated" = "grey70"
)

colorSamples <- c("old"="#7A646A",
                "young"="#80BCEC",
                "all"="#8190a9")

thrMinCells <- 500 * 0.05

In [ ]:
theme_paper <- function(base_size = 17, base_family = "") {
  theme_minimal(base_size = base_size, base_family = base_family) +
    theme(
      plot.title = element_text(face = "bold", size = base_size + 2, hjust = 0.5),
      axis.title = element_text(size = base_size),
      axis.text  = element_text(size = base_size * 0.9),
      legend.title = element_text(size = base_size),
      legend.text  = element_text(size = base_size * 0.9),
      panel.grid.major = element_blank(),
      panel.grid.minor = element_blank(),
      plot.margin = margin(10, 10, 10, 10)
    )
}
theme_set(theme_paper())

In [ ]:
#import workspace
load("workspaces/00_evaluation_objectCreation.Rdata")

In [ ]:
# load truncation annotation generated in /mnt/TEdata/snakemake_TE_full/computeAge_mm10
annotation_truncation <- read.table("annotation/annotated_tes_mm10.tsv", header = TRUE)


In [ ]:
table(annotation_truncation$status)
table(annotation_truncation$annotation)

# Full_Length — the insertion covers ≥90% of the repeat consensus sequence
# 5prime_Truncated — >10% of the consensus is missing from the 5' end only
# 3prime_Truncated — >10% of the consensus is missing from the 3' end only
# Internal_Fragment — >10% missing from both ends; only the middle of the element is present
# Truncated_Other — coverage <90% but neither end gap exceeds 10%; typically a slightly degraded or ambiguously trimmed element

In [ ]:
colnames(annotation_truncation) <- gsub(colnames(annotation_truncation), pattern="genoName", replacement="chr")
colnames(annotation_truncation) <- gsub(colnames(annotation_truncation), pattern="genoStart", replacement="start")
colnames(annotation_truncation) <- gsub(colnames(annotation_truncation), pattern="genoEnd", replacement="end")
annotation_truncation$start <- annotation_truncation$start + 1
colnames(annotation_truncation)

In [ ]:
# merge truncation info to TE annotation
conversion_table <- merge(conversion_table, 
                        annotation_truncation[,c("chr","start","end","strand","coverage","status", "annotation")], 
                        all.x=T)
dim(conversion_table)

# a few loci differ (7k)
sum(is.na(conversion_table$status))

In [ ]:
# add age bin to conversion table
breaks <- c(0, 2, 5, 10, 15, 25, 40, 60, Inf)

conversion_table$age_bin <- cut(
  conversion_table$mya,
  breaks = breaks,
  labels = c("0-2", "2-5", "5-10", "10-15", "15-25", "25-40", "40-60", "60+"),
  right = FALSE,
  include.lowest = TRUE
)

table(conversion_table$age_bin)


In [ ]:
# add finer age bins to conversion table
breaks2 <- c(0, 1, 2, 3, 4, 5, 7, 10, 15, 25, 40, 60, Inf)

conversion_table$age_bin_finer <- cut(
  conversion_table$mya,
  breaks = breaks2,
  labels = c("0-1", "1-2", "2-3", "3-4", "4-5", "5-7", "7-10", "10-15", "15-25", "25-40", "40-60", "60+"),
  right = FALSE,
  include.lowest = TRUE
)

table(conversion_table$age_bin_finer)


## Density maps

In [ ]:
options(repr.plot.width=5, repr.plot.height=4)

layer <- "data"

dfAgeCorr <- NULL

densityPlots <- list()
densityPlotsDf <- NULL

dfCorrList <- list()

# Get density of points in 2 dimensions.
# @param x A numeric vector.
# @param y A numeric vector.
# @param n Create a square n by n grid to compute density.
# @return The density within each square.
get_density <- function(x, y, ...) {
  dens <- MASS::kde2d(x, y,
            h = c(ifelse(bandwidth.nrd(x) == 0, 0.01, bandwidth.nrd(x)),
            ifelse(bandwidth.nrd(x) == 0, 0.01, bandwidth.nrd(y))), ...)
  ix <- findInterval(x, dens$x)
  iy <- findInterval(y, dens$y)
  ii <- cbind(ix, iy)
  return(dens$z[ii])
}

In [ ]:
options(repr.plot.width=5, repr.plot.height=4)
#pdf("figures/densityPlots.pdf")
for(sample in samples){

    splatter_obj <- splatter_objs[[sample]]

    densityPlots[[sample]] <- list()
    dfCorrList[[sample]] <- list()

    for(tool in names(objList)){
        obj <- objList[[tool]][[sample]]

        gc()
        
        TP_TEs <- intersect(Features(splatter_obj),Features(obj))
        
        # extract matrices of TP TE loci from simulated and estimated matrices
        splatter_subset <- GetAssayData(splatter_obj, layer = layer)[TP_TEs,]
        tool_subset <- GetAssayData(obj, layer = layer)[TP_TEs,]

        splatter_df <- reshape2::melt(as.matrix(splatter_subset))
        tool_df <- reshape2::melt(as.matrix(tool_subset))

        df <- splatter_df
        df$tool_value <- tool_df$value
        colnames(df) <- c("TE","Cell", "sim_count", "tool_count")
        
        df$ageClass <- "sample"
        df$order <- conversion_table$class[match(df$TE, conversion_table$stellarscopeID)]
      
        df <- df[rowSums(is.na(df))==0,]
        
        # keep only TP (loci detected and expressed in that cell, both simulated and estimated counts > 0)
        df <- df[((df$sim_count > 0) & (df$tool_count > 0)),] 

        dfCorrList[[sample]][[tool]] <- cor(df$sim_count, df$tool_count, method = "spearman")

        print(paste0("tool ", obj@project.name, ", ", sample, " Spearman correlation: ", 
                    cor(df$sim_count, df$tool_count, method = "spearman")))

        densityPlots[[sample]][[tool]] <- ggplot(df, aes(x=sim_count, y=tool_count)) + 
                geom_hex(bins = 20) +
                scale_fill_viridis(option = "viridis", trans = "sqrt") +
                ggtitle(label = paste0("Spearman correlation = ", round(dfCorrList[[sample]][[tool]],3))) +
                geom_abline(intercept = 0, slope = 1, color="white", linewidth=0.8) + 
                geom_abline(intercept = 0, slope = 1, color="grey40", linewidth=0.5) + 
                xlab("simulated count") +
                ylab(paste0(tool, " count"))+
                # ylim(c(0, max(df_young$sim_count))) +
                theme_minimal() +
                theme(text=element_text(size=18), 
                plot.title = element_text(size=18, hjust=0.5),
                plot.subtitle = element_text(size=18, hjust=0.5))

        show(ggplot(df, aes(x=sim_count, y=tool_count)) + 
                geom_hex(bins = 20) +
                scale_fill_viridis(option = "viridis", trans = "sqrt") +
                ggtitle(paste0(tool, " - " , sample , " TEs"), 
                        subtitle = paste0("Spearman correlation = ", round(dfCorrList[[sample]][[tool]],3))) +
                geom_abline(intercept = 0, slope = 1, color="white", linewidth=0.8) + 
                geom_abline(intercept = 0, slope = 1, color="grey40", linewidth=0.5) + 
                xlab("simulated count") +
                ylab(paste0(tool, " count"))+
                # ylim(c(0, max (df_young$sim_count))) +
                theme_minimal() +
                theme(text=element_text(size=18), 
                plot.title = element_text(size=18, hjust=0.5),
                plot.subtitle = element_text(size=18, hjust=0.5))
            )
        ggsave(paste0("figures_simulated_mm_RA_",sample,"/densityPlot_", TEsubset, "_normalized_",sample, "_", tool, ".pdf"), 
            device="pdf", width=5, height=4)
        
        df$density <- get_density(df$sim_count, df$tool_count, n = 100)
        show(
            ggplot(df) + 
                geom_point(aes(sim_count, tool_count, color = density, alpha = log(density)), size=2) +
                scale_color_viridis(option = "viridis", trans = "sqrt") +
                ggtitle(paste0(tool, " - " , sample , " TEs"), 
                        subtitle = paste0("Spearman correlation = ", round(dfCorrList[[sample]][[tool]], 3))) +
                geom_abline(intercept = 0, slope = 1, color="white", linewidth=0.8) + 
                geom_abline(intercept = 0, slope = 1, color="grey40", linewidth=0.5) + 
                scale_alpha_continuous(range = c(0.3, 1)) +
                xlab("simulated count") +
                ylab(paste0(tool, " count")) +
                guides(alpha = "none") +
                theme_minimal() +
                theme(text=element_text(size=18), 
                plot.title = element_text(size=18, hjust=0.5),
                plot.subtitle = element_text(size=18, hjust=0.5))
            )
        ggsave(paste0("figures_simulated_mm_RA_",sample,"/densityScatterPlot_", TEsubset, "_normalized_",sample, "_", tool, ".png"), 
            device="png", width=5, height=4, dpi=600)
        ggsave(paste0("figures_simulated_mm_RA_",sample,"/densityScatterPlot_", TEsubset, "_normalized_",sample, "_", tool, ".pdf"), 
            device="pdf", width=5, height=4)

        # create df with all the data points
        densityPlotsDf <- rbind(densityPlotsDf, cbind(df, tool, sample))
    }
}


In [ ]:
save.image(paste0("workspaces/", dataset_id, "_levels_density_maps_", TEsubset,".Rdata"))

### Checkpoint

In [ ]:
load(paste0("workspaces/", dataset_id, "_levels_density_maps_", TEsubset,".Rdata"))

In [ ]:
gc()

In [ ]:
options(repr.plot.width=20, repr.plot.height=4)
gc()

for(sample in samples){
    show(
        ggplot(densityPlotsDf[densityPlotsDf$sample==sample,]) + 
            geom_point(aes(sim_count, tool_count, color = density, alpha=density), size=2) +
            facet_grid(rows = vars(sample), cols = vars(tool)) +
            scale_color_viridis(option = "viridis", trans = "sqrt") +
            geom_abline(intercept = 0, slope = 1, color="white", linewidth=0.8) + 
            geom_abline(intercept = 0, slope = 1, color="grey40", linewidth=0.5) + 
            scale_alpha_continuous(range = c(0.3, 1)) +
            xlab("simulated normalized count") +
            ylab(paste0("inferred normalized count")) +
            guides(alpha = "none") +
            theme_light() +
            theme(text=element_text(size=18), strip.text=element_text(size=18))
    )
    ggsave(paste0("figures_simulated_mm_RA_",sample,"/grid_scatter_density_plots_facet_", TEsubset,".pdf"), device="pdf",width = 20, height = 4)
}

In [ ]:
options(repr.plot.width=20, repr.plot.height=4)
gc()

for(sample in samples){
    show(
        ggplot(densityPlotsDf[densityPlotsDf$sample==sample,], aes(x=sim_count, y=tool_count)) + 
            facet_grid(rows = vars(sample), cols = vars(tool)) +
            geom_hex(bins = 30) +
            scale_fill_viridis(option = "viridis", trans="sqrt") +
            #ggtitle(paste0(tool, " - " , sample , " TEs")) +
            geom_abline(intercept = 0, slope = 1, color="white", linewidth=1) +
            geom_abline(intercept = 0, slope = 1, color="grey40", linewidth=0.8) +
            xlab("simulated normalized count") +
            ylab(paste0("inferred normalized count"))+
            theme_light() +
            theme(text=element_text(size=18), strip.text=element_text(size=18))
    )
    ggsave(paste0("figures_simulated_mm_RA_",sample,"/grid_density_hex_plots_facet_",TEsubset,".pdf"), device="pdf",width = 20, height = 4)
}

## Metrics by locus

In [ ]:
densityPlotsDf$family <- conversion_table$family[match(densityPlotsDf$TE, conversion_table$stellarscopeID)]
densityPlotsDf$subfamily <- conversion_table$subfamily[match(densityPlotsDf$TE, conversion_table$stellarscopeID)]
densityPlotsDf$propSubs <- conversion_table$propSubs[match(densityPlotsDf$TE, conversion_table$stellarscopeID)]

## Compute correlations

In [ ]:
library(dplyr)

# group by TE and by tool and compute spearman correlation, only for loci detected in at least 3 cells

cor_by_TE <- densityPlotsDf %>%
  group_by(TE, tool, sample) %>%
  filter(n() >= thrMinCells) %>%
  summarise(
    order  = first(order),
    family = first(family),
    subfamily = first(subfamily),
    n = n(),
    cor = cor(sim_count, tool_count, method="spearman", use = "complete.obs"),
    p_value = cor.test(sim_count, tool_count, method = "spearman", use = "complete.obs")$p.value,
    .groups = "drop"
  )


In [ ]:
# order tools to plot
cor_by_TE$tool <- factor(cor_by_TE$tool , levels = rev(names(colorTools)))

# add other covariates
cor_by_TE$mya <- conversion_table$mya[match(cor_by_TE$TE, conversion_table$stellarscopeID)]
cor_by_TE$propSubs <- conversion_table$propSubs[match(cor_by_TE$TE, conversion_table$stellarscopeID)]
cor_by_TE$TElength <- conversion_table$end[match(cor_by_TE$TE, conversion_table$stellarscopeID)] - 
                                        conversion_table$start[match(cor_by_TE$TE, conversion_table$stellarscopeID)] 

cor_by_TE$age_bin <- conversion_table$age_bin[match(cor_by_TE$TE, conversion_table$stellarscopeID)]
cor_by_TE$age_bin_finer <- conversion_table$age_bin_finer[match(cor_by_TE$TE, conversion_table$stellarscopeID)]
cor_by_TE$truncationStatus <- conversion_table$status[match(cor_by_TE$TE, conversion_table$stellarscopeID)]
cor_by_TE$truncationAnnotation <- conversion_table$annotation[match(cor_by_TE$TE, conversion_table$stellarscopeID)]


head(cor_by_TE)

In [ ]:
#library(ggbeeswarm)

options(repr.plot.width=6.5, repr.plot.height=5)

sample <- "young"

ggplot(cor_by_TE[cor_by_TE$sample == sample,]) + aes(x=cor, fill=tool,  y=tool) + 
  facet_wrap(~sample) +
  geom_violin( linewidth = 0.5, alpha = 0.8 ) + 
  geom_point(position = position_jitter(), size=0.3, alpha=0.5, color="grey30") +
#   geom_beeswarm(size=0.05, alpha = 0.5) +
  # geom_dotplot(binaxis= "x",
  #              stackdir = "center",
  #              dotsize = 0.05,
  #              fill = 1) +
  ggtitle("Correlation between inferred and", 
          subtitle = " simulated counts of common TEs\n") +
  xlab("Spearman cor. coef.") +
  scale_fill_manual(values = colorTools) +
  theme_minimal() + 
  theme(text=element_text(size=20), axis.text.y = element_text(size=18),
        plot.title = element_text(size=18, hjust=0.5), 
        plot.subtitle = element_text(size=18, hjust=0.5), 
        strip.text = element_text(size=18, hjust=0.5, vjust = 1 ), 
        strip.background = element_rect(fill=alpha(colorSamples[sample], 0.5), color = "white"),
        panel.spacing = unit(3, "lines")) + NoLegend()

ggsave(paste0("figures_simulated_mm_RA_",sample,"/correlation_perTE_violin_jitter",sample,"_byTool.pdf"))

ggplot(cor_by_TE[cor_by_TE$sample == sample,]) + aes(x=cor, fill=tool,  y=tool) + 
  facet_wrap(~sample) +
  geom_violin( linewidth = 0.5, alpha = 0.8 ) + 
  ggtitle("Correlation between inferred and", 
          subtitle = " simulated counts of common TEs\n") +
  xlab("Spearman cor. coef.") +
  scale_fill_manual(values = colorTools) +
  theme_minimal() + 
  theme(text=element_text(size=20), axis.text.y = element_text(size=20),
        plot.title = element_text(size=18, hjust=0.5), 
        plot.subtitle = element_text(size=18, hjust=0.5), 
        strip.text = element_text(size=20, hjust=0.5, vjust = 1 ), 
        strip.background = element_rect(fill=alpha(colorSamples[sample], 0.5), color = "white"),
        panel.spacing = unit(3, "lines")) + NoLegend()

ggsave(paste0("figures_",dataset_id,"_",sample,"/correlation_perTE_violin_",sample,"_byTool.pdf"),width=6.5, height=5)


options(repr.plot.width=11, repr.plot.height=5)
ggplot(cor_by_TE[cor_by_TE$sample %in% c("old","young"),]) + aes(x=cor, fill=tool,  y=tool) + 
  facet_wrap(~sample) +
  geom_boxplot( linewidth = 0.5, alpha = 0.8 ) + 
  ggtitle("Correlation between inferred and", 
          subtitle = " simulated counts of common TEs\n") +
  xlab("Spearman cor. coef.") +
  scale_fill_manual(values = colorTools) +
  theme_minimal() + 
  theme(text=element_text(size=20), axis.text.y = element_text(size=20),
        plot.title = element_text(size=18, hjust=0.5), 
        plot.subtitle = element_text(size=18, hjust=0.5), 
        strip.text = element_text(size=20, hjust=0.5, vjust = 1 ), 
        #strip.background = element_rect(fill=alpha(colorSamples[sample], 0.5), color = "white"),
        panel.spacing = unit(3, "lines")) + NoLegend()

ggsave(paste0("figures_",dataset_id,"_",sample,"/correlation_perTE_boxplot_",sample,"_byAge_byTool.pdf"),width=11, height=5)

In [ ]:
#Test bimodality
library(diptest)
dip.test(cor_by_TE[cor_by_TE$sample == "young", ]$cor)


## Test predictors of quantification accuracy

In [ ]:
# add average expression 
layer <- "data"
avgExprPerExprCellList <- list()

for(sample in samples){
    splatter_obj <- splatter_objs[[sample]]
    mat_splatter <- GetAssayData(splatter_obj, layer = layer)
    # compute average expression in the cells where each TE is expressed
    avgExpr<- rowMeans(mat_splatter) # avg across cells
    sumExpr <- rowSums(mat_splatter)
    nCellsExpr <- rowSums(mat_splatter > 0)
    avgExprPerExprCell <- sumExpr/nCellsExpr # avg for cells where the TE is expressed
    avgExprPerExprCellList[[sample]] <- avgExprPerExprCell
}

In [ ]:
cor_by_TE$tool <- factor(cor_by_TE$tool, levels=unique(cor_by_TE$tool))
table(cor_by_TE$tool)

### Aggregate "other" and "internal" truncations

In [ ]:
table(cor_by_TE$truncationAnnotation)

cor_by_TE$truncationType <- gsub(cor_by_TE$truncationAnnotation, pattern="Internal_Fragment", replacement="Truncated_Other")
table(cor_by_TE$truncationType)

### GEE

In [ ]:

sample <- "young"

# select correlations of young TEs
cor_by_TE_Young <- cor_by_TE[cor_by_TE$sample == sample, ]

# add avg expression
cor_by_TE_Young$avgExpr <- avgExprPerExprCellList[[sample]][as.character(cor_by_TE_Young$TE)]

# Create binary outcome
cor_by_TE_logit <- cor_by_TE_Young %>%
  mutate(high_accuracy = ifelse(cor > median(cor), 1, 0))

# scale
cor_by_TE_logit2 <- cor_by_TE_logit %>%
  mutate(
    across(c(propSubs, TElength, avgExpr), ~as.numeric(scale(.x))),
    truncationType = factor(truncationType, 
      levels = c('Full_Length', '5prime_Truncated','3prime_Truncated', 'Truncated_Other'))
  )

gee_model <- geeglm(
  high_accuracy ~ tool + TElength + avgExpr + propSubs + truncationType,
  id = TE,   # cluster by locus
  data = cor_by_TE_logit2,
  family = binomial,
  corstr = "exchangeable"  # assumes equal correlation among repeated obs within a locus
)
summary(gee_model)


In [ ]:

# 1. Tidy the GEE model and manually construct CIs (broom doesn't auto-compute CIs for geeglm)
or_tab <- broom::tidy(gee_model) %>%
  mutate(
    conf.low  = estimate - 1.96 * std.error,
    conf.high = estimate + 1.96 * std.error,
    # exponentiate estimate and CI bounds to get odds ratios
    estimate  = exp(estimate),
    conf.low  = exp(conf.low),
    conf.high = exp(conf.high)
  )

# 2. Clean the dataset: remove intercept, remove tools, and clean up labels
or_plot_data <- or_tab %>%
  filter(term != "(Intercept)") %>%
  filter(!grepl("^tool", term)) %>% # Dynamically drops all tool rows
  mutate(
    # Clean up the ugly R factor prefixes for the plot labels
    term = gsub("truncationAnnotation", "", term),
    # Reorder terms based on the odds ratio estimate value
    term = reorder(term, estimate)
  )

# 3. Plotting
options(repr.plot.width=8, repr.plot.height=4)

ggplot(or_plot_data, aes(y = term, x = estimate, xmin = conf.low, xmax = conf.high)) +
  geom_point(size = 3, color = "grey20") +
  geom_errorbarh(height = 0.15, color = "grey20", linewidth = 0.8) +
  geom_vline(xintercept = 1, linetype = "dashed", color = "grey60", linewidth = 0.7) +
  scale_x_continuous(transform = "log10") +
  labs(
    x = "Odds Ratio (per 1 SD increase for continuous variables)",
    y = ""
  ) +
  theme_pubr() +
  theme(
    text = element_text(size = 15),
    axis.text.y = element_text(face = "bold")
  )

# 4. Save
ggsave(paste0("figures_simulated_mm_RA_", sample, "/OR_locus_toolfixed_GEE_features_truncationtypes.pdf"),
       device = "pdf", width = 9, height = 5)

### Tool interaction

In [ ]:
gee_interaction <- geeglm(
  high_accuracy ~ tool * (TElength + avgExpr + propSubs + truncationType),
  id = TE, data = cor_by_TE_logit2, family = binomial, corstr = "exchangeable"
)

QIC(gee_model)
QIC(gee_interaction)

summary(gee_interaction)

# Families

In [ ]:
unique(cor_by_TE[cor_by_TE$sample == sample,]$family)

In [ ]:
options(repr.plot.width=6, repr.plot.height=5)

sample <- "young"

# define order for plotting
familyOrder <- c("L1",'ERVL','ERVK','ERV1','ERVL-MaLR','B2','Alu')
df <- cor_by_TE[cor_by_TE$sample == sample,]
df$family <- factor(df$family, levels = rev(familyOrder))

ggplot(df) + 
        aes(x=cor, fill=family,  y=family) + 
  facet_wrap(~sample) +
  geom_violin( linewidth = 0.5, alpha = 0.7 ) + 
  geom_point(position = position_jitter(), size=0.5, alpha=0.5, color="grey30") +
#   geom_beeswarm(size=0.05, alpha = 0.5) +
  # geom_dotplot(binaxis= "x",
  #              stackdir = "center",
  #              dotsize = 0.05,
  #              fill = 1) +
  ggtitle("Correlation between inferred and", 
          subtitle = " simulated counts of common TEs\n") +
  xlab("Spearman cor. coef.") +
  scale_fill_scico_d(palette="lipari") +
  #scale_fill_brewer(palette="Spectral") +
  theme_minimal() + 
  theme(text=element_text(size=20), axis.text.y = element_text(size=18),
        plot.title = element_text(size=18, hjust=0.5), 
        plot.subtitle = element_text(size=18, hjust=0.5), 
        strip.text = element_text(size=18, hjust=0.5, vjust = 1 ), 
        strip.background = element_rect(fill=alpha(colorSamples[sample], 0.5), color = "white"),
        panel.spacing = unit(3, "lines")) + NoLegend()

ggsave(paste0("figures_simulated_mm_RA_",sample,"/correlation_perTE_violin_jitter",sample,"_", TEsubset,".pdf"),width=6, height=5)


ggplot(df) + aes(x=cor, fill=family,  y=family) + 
  facet_wrap(~sample) +
  geom_violin( linewidth = 0.5, alpha = 0.7 ) + 
  ggtitle("Correlation between inferred and", 
          subtitle = " simulated counts of common TEs\n") +
  xlab("Spearman cor. coef.") +
  scale_fill_scico_d(palette="lipari") +
  #scale_fill_brewer(palette="Spectral") +
  theme_minimal() + 
  theme(text=element_text(size=20), axis.text.y = element_text(size=20),
        plot.title = element_text(size=18, hjust=0.5), 
        plot.subtitle = element_text(size=18, hjust=0.5), 
        strip.text = element_text(size=20, hjust=0.5, vjust = 1 ), 
        strip.background = element_rect(fill=alpha(colorSamples[sample], 0.5), color = "white"),
        panel.spacing = unit(3, "lines")) + NoLegend()

ggsave(paste0("figures_",dataset_id,"_",sample,"/correlation_perTE_violin_",sample,"_", TEsubset,"_byFamily.pdf"),width=6, height=5)

# Overall metrics

## TP counts

In [ ]:

# r squared
rsq <- function(x, y) summary(lm(y~x))$r.squared
rsqList <- list()


# root mean square error
rmse <- function(actual, predicted) sqrt(mean((actual - predicted)^2))
rmseList <- list()

# correlation
corList <- list()

In [ ]:
layer <- "data"

TPcountsDfs <- list()

for(sample in samples){


    print(sample)

    rsqList[[sample]] <- list()
    rmseList[[sample]] <- list()
    corList[[sample]] <- list()

    splatter_obj <- splatter_objs[[sample]]

    for(tool in names(objList)){
        print(tool)
        
        obj <- objList[[tool]][[sample]]

        gc()

        TP_TEs <- intersect(Features(splatter_obj), Features(obj)) # TP TEs
        
        # extract matrices of TP TE loci from simulated and estimated matrices
        splatter_subset <- GetAssayData(splatter_obj, layer = layer)[TP_TEs,]
        tool_subset <- GetAssayData(obj, layer = layer)[TP_TEs,]
        cat("Same row names: ", identical(rownames(splatter_subset), rownames(tool_subset)), "\n")
        cat("Same col names: ", identical(colnames(splatter_subset), colnames(tool_subset)), "\n")
        
        print(dim(splatter_subset))

        splatter_df <- reshape2::melt(as.matrix(splatter_subset))
        tool_df <- reshape2::melt(as.matrix(tool_subset))

        df <- splatter_df
        df$tool_value <- tool_df$value
        colnames(df) <- c("TE","Cell", "sim_count", "tool_count")
        
        df$ageClass <- sample
      
        #df <- df[rowSums(is.na(df))==0,]
        
        # keep only TP (loci detected and expressed in that cell, both simulated and estimated counts > 0)
        df <- df[((df$sim_count > 0) & (df$tool_count > 0)),] 
        
        # add age and family info
        #df$ageClass <- conversion_table$ageClass[match(df$TE, conversion_table$stellarscopeID)]
        df$order <- conversion_table$class[match(df$TE, conversion_table$stellarscopeID)]
        df$family <- conversion_table$family[match(df$TE, conversion_table$stellarscopeID)]
        df$subfamily <- conversion_table$subfamily[match(df$TE, conversion_table$stellarscopeID)]
        
        # Compute R2
        rsqList[[sample]][[tool]] <- rsq(df$tool_count, df$sim_count)
  
        # Compute RMSE
        rmseList[[sample]][[tool]] <- rmse(df$sim_count, df$tool_count)

        # Compute spearman correlation 
        corList[[sample]][[tool]] <- cor(df$sim_count, df$tool_count, method="spearman")

        TPcountsDfs[[sample]][[tool]] <- df
    }
}

In [ ]:
cor_by_tool <- densityPlotsDf %>%
  group_by(tool, sample) %>%
  filter(n() >= 3) %>%
  summarise(
    n = n(),
    cor = cor(sim_count, tool_count, method="spearman", use = "complete.obs"),
    p_value = cor.test(sim_count, tool_count, method = "spearman", use = "complete.obs")$p.value,
    .groups = "drop"
  )

cor_by_tool[cor_by_tool$sample == "young",]


In [ ]:
cor_by_family <- densityPlotsDf %>%
  group_by(tool, family, sample) %>%
  filter(n() >= 3) %>%
  summarise(
    n = n(),
    cor = cor(sim_count, tool_count, method="spearman", use = "complete.obs"),
    p_value = cor.test(sim_count, tool_count, method = "spearman", use = "complete.obs")$p.value,
    .groups = "drop"
  )

cor_by_family[cor_by_family$sample == "young",]


In [ ]:
summary(cor_by_tool[cor_by_tool$sample == "young","cor"])
summary(cor_by_TE[cor_by_TE$sample == "young","cor"])

In [ ]:
options(repr.plot.width=6.5, repr.plot.height=4)
colorSamples <- c("old"="#7A646A",
                "young"="#80BCEC",
                "all"="#8190a9")

# plot R squared
df_old <- cbind(melt(rsqList[["old"]]),"old")
colnames(df_old) <- c("Rsq","tool","age")
df_young <- cbind(cbind(melt(rsqList[["young"]]), "young"))
colnames(df_young) <- c("Rsq","tool","age")
df <- rbind(df_old, df_young)
df$tool <- factor(df$tool, levels=rev(names(colorTools)))

ggplot(df, aes(x=Rsq, y=tool, fill=age)) + 
   geom_col(
    position = position_dodge2(width = 0.9, padding = 0.1), 
    width = 0.85,
    alpha = 1
  ) +
  ylab("") +
  scale_fill_manual(values=colorSamples) +
  theme_pubclean() + ggtitle(paste0("R-squared")) +
  theme(text=element_text(size=20),
        axis.text.y = element_text(size=18),
        axis.label = element_text(size=18),
        legend.title = element_text(size=18),
        legend.text = element_text(size=18),
        plot.title = element_text(size=20, hjust=0.5))
ggsave(paste0("figures_", dataset_id, "/rsquared_barplot_",TEsubset,".pdf"), device="pdf", width=6.5, height=4)

# plot root Mean Square Error
df_old <- cbind(melt(rmseList[["old"]]),"old")
colnames(df_old) <- c("RMSE","tool","age")
df_young <- cbind(cbind(melt(rmseList[["young"]]), "young"))
colnames(df_young) <- c("RMSE","tool","age")
df <- rbind(df_old, df_young)
df$tool <- factor(df$tool, levels=rev(names(colorTools)))

ggplot(df, aes(x=RMSE, y=tool, fill=age)) + 
   geom_col(
    position = position_dodge2(width = 0.9, padding = 0.1), 
    width = 0.85,
    alpha = 1
  ) +
  ylab("") +
  scale_fill_manual(values=colorSamples) +
  theme_pubclean() + 
  ggtitle(paste0("Root-Mean-Square Error")) +
    theme(text=element_text(size=20),
        axis.text.y = element_text(size=19),
        legend.title = element_text(size=18),
        legend.text = element_text(size=18),
        plot.title = element_text(size=20, hjust=0.5))
ggsave(paste0("figures_", dataset_id, "/rmse_barplot_",TEsubset,".pdf"), device="pdf", width=6.5, height=4)


# plot spearman correlation
df_old <- cbind(melt(corList[["old"]]),"old")
colnames(df_old) <- c("Correlation","tool","age")
df_young <- cbind(cbind(melt(corList[["young"]]), "young"))
colnames(df_young) <- c("Correlation","tool","age")
df <- rbind(df_old, df_young)
df$tool <- factor(df$tool, levels=rev(names(colorTools)))

ggplot(df, aes(x=Correlation, y=tool, fill=age)) + 
   geom_col(
    position = position_dodge2(width = 0.9, padding = 0.1), 
    width = 0.85,
    alpha = 1
  ) +
  ylab("") +
  scale_fill_manual(values=colorSamples) +
  theme_pubclean() + 
  ggtitle(paste0("Correlation")) +
  xlab("Spearman cor. coef.") +
    theme(text=element_text(size=20),
        axis.text.y = element_text(size=19),
        legend.title = element_text(size=18),
        legend.text = element_text(size=18),
        plot.title = element_text(size=20, hjust=0.5))

ggsave(paste0("figures_", dataset_id, "/spearmanCor_barplot_",TEsubset,".pdf"), device="pdf", width=6.5, height=4)

write.table(df, paste0("data/", dataset_id, "_correlation_TPcounts.tsv"))